# 06 — Decision models compared on the same state

**Question this notebook answers:** what sits in the "decision model"
slot of the chain, and how do the options differ on identical input?

The chain is agnostic to the decision model. Three implementations exist:

| Model | Transport | Notes |
| --- | --- | --- |
| Rust Laya | local daemon, JSONL | open-source, CPU, ~0.6 s/decision |
| TypeSafe Jev | REST `POST /v1/systemone` | hosted, needs `TYPESAFE_API_KEY` |
| mock | in-process | deterministic fallback, no model |

This notebook sends the **same** state and questions to Laya and (if the
key is present) to Jev, and prints the decisions side by side.


In [ ]:
import os, sys

# make the `lab` package importable whether jupyter starts here or in the repo root
for _p in (os.getcwd(), os.path.dirname(os.getcwd())):
    if os.path.exists(os.path.join(_p, "lab")):
        sys.path.insert(0, _p)
        PROJ = _p
        break
else:
    raise RuntimeError("run jupyter from the project root or the notebooks/ directory")

from lab import EwmScene, LayaDaemon, BonsaiClient
print("project root:", PROJ)



In [ ]:
import os

STATE = "The user asked: \"Explain gravity in one sentence.\" The system's memory holds 24 content-addressed token ids from earlier answers."
OPTIONS = {
    "full": "keep the full materialized memory as the context",
    "compact": "keep only new, previously unseen tids as the context",
    "surprise": "keep only structurally surprising content",
}
INSTRUCTIONS = "How should the next context be built for the base model?"


## Laya (local daemon)


In [ ]:
laya = LayaDaemon()
d_laya = laya.route(STATE, OPTIONS, INSTRUCTIONS, extra_noul={"keep_short": "The context should be kept short"}, verbose=True)
laya.close()


## Jev (TypeSafe REST) — only when the key is present

The request shape is identical in spirit; the wire format is the TypeSafe
System One REST API.


In [ ]:
try:
    import sys as _sys
    sys.path.insert(0, os.path.join(PROJ, "..", "bonsai-ewm", "src"))
    from bonsai_ewm.adapters import JevAdapter
    jev = JevAdapter()
    if jev.mock:
        print("Jev: mock (TYPESAFE_API_KEY not set) — skipping real call")
    else:
        r = jev.route(
            {"turn": 2, "query": "Explain gravity in one sentence.",
             "union_pop": 18, "full_memory_tids": 24, "novelty_tids": 9,
             "last_answer_chars": 120},
            OPTIONS, INSTRUCTIONS, extra_noul={"keep_short": "The context should be kept short"},
        )
        print("Jev decision:", r["decision"])
        print("Jev probabilities:", r["probabilities"])
        print("Jev confidence:", round(r["confidence"], 4))
        print("Jev keep_short:", round(r["aux"]["keep_short"], 4))
        print("Jev model:", r["model"], "| input_tokens:", r["input_tokens"])
except Exception as exc:
    print("Jev call skipped:", exc)


## Mock (deterministic, no model)

The mock exists so the chain runs without any decision model. It is
deterministic in the query, not intelligent — useful for tests, not for
real routing.


In [ ]:
import zlib
seed = zlib.crc32(STATE.encode())
names = list(OPTIONS)
vals = [1.0 + ((seed >> i) & 3) for i in range(len(names))]
total = sum(vals)
probs = {n: v / total for n, v in zip(names, vals)}
print("mock decision:", max(probs, key=probs.get))
print("mock probabilities:", {k: round(v, 4) for k, v in probs.items()})
print("mock keep_short:", 0.5)


## Summary

- All three fill the same slot: `(state, options, instructions, noul) →
  decision + confidence + probabilities`.
- Laya is local and ungated — the default for the chain in this project.
- Jev is the hosted sibling; the bonsai-ewm production controller uses it
  in `auto` mode when `TYPESAFE_API_KEY` is set.
- The mock keeps offline development moving and must never be mistaken
  for a real decision.
